In [1]:
import pandas as pd
import glob
import plotly.express as px
import plotly.io as pio

# ตั้งค่า renderer ให้เหมาะกับ VS Code
pio.renderers.default = "browser"

# โหลดไฟล์ CSV ทุกไฟล์ที่ชื่อขึ้นต้นด้วย flightsfrom_BKK_
files = glob.glob("flightsfrom_HDY_*.csv")

# รวมข้อมูลทุกไฟล์
df_list = [pd.read_csv(f) for f in files]
df = pd.concat(df_list, ignore_index=True)

# ทำให้ชื่อคอลัมน์เป็นตัวเล็กและลบช่องว่าง
df.columns = df.columns.str.strip().str.lower()

# สรุปจำนวนเที่ยวบินต่อวัน
summary = df.groupby(['date','direction']).size().reset_index(name='count')
pivot = summary.pivot(index='date', columns='direction', values='count').fillna(0)

# เพิ่มคอลัมน์ diff (Dept. - Arriv.)
pivot['diff'] = pivot['departure'] - pivot['arrival']

# -------------------------------
# ส่วนที่ 1: กราฟ (ทุกวันตามปกติ, ไม่มีเส้น Total)
# -------------------------------
fig = px.line(
    pivot,
    x=pivot.index,
    y=['departure','arrival'],
    labels={'value':'จำนวนเที่ยวบิน (เที่ยว)','date':'วันที่'},
    title="สถิติเที่ยวบินรายวัน (BKK) path : ..\ search-flight-27\ backend\data\ asia\ thailand"
)

fig.update_traces(mode="lines+markers")
fig.update_layout(hovermode="x unified")
fig.show()

# -------------------------------
# ส่วนที่ 2: คำนวณสัดส่วนร้อยละของวันที่ต่างกันเกิน 100
# -------------------------------
def calc_damage_ratio(dataframe, threshold=100):
    total_days = len(dataframe)
    damaged_days = (abs(dataframe['diff']) > threshold).sum()
    ratio = (damaged_days / total_days) * 100
    print(f"จำนวนวันทั้งหมด: {total_days}")
    print(f"จำนวนวันที่ Dept. - Arriv. ห่างกันเกิน {threshold}: {damaged_days}")
    print(f"คิดเป็น {ratio:.2f}% ของข้อมูลทั้งหมด")

# เรียกใช้งานฟังก์ชัน
calc_damage_ratio(pivot, threshold=100)

<>:34: SyntaxWarning: invalid escape sequence '\ '
<>:34: SyntaxWarning: invalid escape sequence '\ '
C:\Users\Victus\AppData\Local\Temp\ipykernel_27464\2242716635.py:34: SyntaxWarning: invalid escape sequence '\ '
  title="สถิติเที่ยวบินรายวัน (BKK) path : ..\ search-flight-27\ backend\data\ asia\ thailand"


จำนวนวันทั้งหมด: 323
จำนวนวันที่ Dept. - Arriv. ห่างกันเกิน 100: 0
คิดเป็น 0.00% ของข้อมูลทั้งหมด


In [4]:
import pandas as pd
import glob
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly.graph_objects as go

pio.renderers.default = "browser"

# -------------------------------
# ฟังก์ชันจากโค้ดที่สอง
# -------------------------------
def _find_csv_files(pattern):
    files = glob.glob(pattern, recursive=True)
    return [f for f in files if f.lower().endswith(".csv")]

def _read_all_csv(files):
    df_list = []
    for f in files:
        try:
            df = pd.read_csv(f)
            df["__source_file"] = f
            df_list.append(df)
        except Exception as e:
            print(f"[skip] read failed: {f} | {e}")
    if not df_list:
        return pd.DataFrame()
    return pd.concat(df_list, ignore_index=True)

def _normalize(df):
    df.columns = df.columns.str.strip().str.lower()
    if "airport" in df.columns:
        df["airport"] = df["airport"].astype(str).str.strip().str.upper()
    if "direction" in df.columns:
        df["direction"] = df["direction"].astype(str).str.strip().str.lower()
    if "date" in df.columns:
        dt = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
        df["date_dt"] = dt
        df["date_key"] = dt.dt.strftime("%Y-%m-%d")
    return df

def build_daily_pivot(df_one_airport):
    summary = (
        df_one_airport.groupby(["date_key", "direction"])
        .size()
        .reset_index(name="count")
    )
    pivot = (
        summary.pivot(index="date_key", columns="direction", values="count")
        .fillna(0)
        .sort_index()
    )
    if "arrival" not in pivot.columns:
        pivot["arrival"] = 0
    if "departure" not in pivot.columns:
        pivot["departure"] = 0
    pivot["diff"] = pivot["departure"] - pivot["arrival"]
    return pivot

def calc_damage_ratio(pivot_df, threshold=100):
    total_days = len(pivot_df)
    if total_days == 0:
        return 0, 0, 0.0
    damaged_days = (pivot_df["diff"].abs() > threshold).sum()
    ratio = (damaged_days / total_days) * 100
    print(f"จำนวนวันทั้งหมด: {total_days}")
    print(f"จำนวนวันที่ Dept. - Arriv. ห่างกันเกิน {threshold}: {damaged_days}")
    print(f"คิดเป็น {ratio:.2f}% ของข้อมูลทั้งหมด")
    return total_days, damaged_days, ratio

# -------------------------------
# ใช้งานแบบโค้ดแรก (สนามบินเดียว)
# -------------------------------
files = _find_csv_files("flightsfrom_HKT_*.csv")
df = _read_all_csv(files)
df = _normalize(df)

pivot = build_daily_pivot(df[df["airport"] == "HKT"])

fig = px.line(
    pivot,
    x=pivot.index,
    y=["departure","arrival"],
    labels={'value':'จำนวนเที่ยวบิน (เที่ยว)','date_key':'วันที่'},
    title="สถิติเที่ยวบินรายวัน (HKT)"
)
fig.update_traces(mode="lines+markers")
fig.update_layout(hovermode="x unified")
fig.show()

calc_damage_ratio(pivot, threshold=100)

จำนวนวันทั้งหมด: 121
จำนวนวันที่ Dept. - Arriv. ห่างกันเกิน 100: 23
คิดเป็น 19.01% ของข้อมูลทั้งหมด


(121, np.int64(23), np.float64(19.00826446280992))

In [19]:
import pandas as pd
import glob
import plotly.express as px

# -----------------------------
# Load data
# -----------------------------
files = glob.glob("flightsfrom_*_*.csv")

df_list = [pd.read_csv(f) for f in files]
df = pd.concat(df_list, ignore_index=True)

df.columns = df.columns.str.strip().str.lower()
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.to_period('M').astype(str)

# -----------------------------
# helper : auto chart height
# -----------------------------
def auto_height(n_rows, base=40, min_h=320, max_h=800):
    h = base * n_rows
    return max(min_h, min(h, max_h))

#Tab 1 Market & Routes

Flight Demand Trend

In [20]:
daily = df.groupby('date').size().reset_index(name='flights')

fig = px.line(
    daily,
    x='date',
    y='flights',
    title='Flight Demand Trend',
    markers=True
)

fig.update_layout(height=400)

fig.show()

Top Routes by Flights (Horizontal Ranking)

In [23]:
routes = (
    df.groupby('destination')
      .size()
      .reset_index(name='flights')
      .sort_values('flights', ascending=True)
)

h = auto_height(len(routes))

fig = px.bar(
    routes,
    x='flights',
    y='destination',
    orientation='h',
    title='Top Routes by Number of Flights',
    text_auto=True
)

fig.update_layout(height=h)

fig.show()

Route Competition (Airlines per Route)

In [24]:
competition = (
    df.groupby('destination')['airline']
      .nunique()
      .reset_index(name='airline_count')
      .sort_values('airline_count', ascending=True)
)

h = auto_height(len(competition))

fig = px.bar(
    competition,
    x='airline_count',
    y='destination',
    orientation='h',
    title='Route Competition (Number of Airlines)',
    text_auto=True
)

fig.update_layout(height=h)

fig.show()

#Tab 2 Airline Landscape

Top Airlines by Flights

In [25]:
airline_summary = (
    df.groupby('airline')
      .size()
      .reset_index(name='flights')
      .sort_values('flights', ascending=True)
)

h = auto_height(len(airline_summary))

fig = px.bar(
    airline_summary,
    x='flights',
    y='airline',
    orientation='h',
    title='Top Airlines by Number of Flights',
    text_auto=True
)

fig.update_layout(height=h)

fig.show()

Airline Activity Trend

In [26]:
trend = (
    df.groupby(['month','airline'])
      .size()
      .reset_index(name='flights')
)

fig = px.line(
    trend,
    x='month',
    y='flights',
    color='airline',
    title='Airline Activity Trend'
)

fig.update_layout(height=420)

fig.show()

Airline Route Coverage

In [27]:
coverage = (
    df.groupby('airline')['destination']
      .nunique()
      .reset_index(name='routes')
      .sort_values('routes', ascending=True)
)

h = auto_height(len(coverage))

fig = px.bar(
    coverage,
    x='routes',
    y='airline',
    orientation='h',
    title='Airline Route Coverage',
    text_auto=True
)

fig.update_layout(height=h)

fig.show()

In [29]:
market = (
    df.groupby('airline')
      .size()
      .reset_index(name='flights')
)

fig = px.pie(
    market,
    values='flights',
    names='airline',
    title='Airline Market Share'
)

fig.show()